In [1]:
!uv add lightgbm

Resolved 201 packages in 38ms
Audited 198 packages in 107ms


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 데이터 로드
df = pd.read_csv('preprocessed_shelter_data.csv')

# 보호중 제외 (target >= 0)
df_model = df[df['target'] >= 0].copy()

print(f"총 데이터: {len(df_model)}건")
print(f"\n타겟 분포:")
print(df_model['target'].value_counts().sort_index())
print(f"\n클래스 비율:")
print(df_model['target'].value_counts(normalize=True).sort_index())

총 데이터: 2650건

타겟 분포:
target
0    1612
1     332
2     706
Name: count, dtype: int64

클래스 비율:
target
0    0.608302
1    0.125283
2    0.266415
Name: proportion, dtype: float64


In [3]:
# 제외할 컬럼 (메타데이터, 원본 텍스트)
exclude_cols = ['desertionNo', 'happenDt', 'processState', 'specialMark']

# 타겟 분리
X = df_model.drop(['target'] + exclude_cols, axis=1)
y = df_model['target']

print(f"피처 수: {X.shape[1]}개")
print(f"샘플 수: {X.shape[0]}건")

피처 수: 20개
샘플 수: 2650건


In [4]:
from sklearn.preprocessing import LabelEncoder

# 카테고리 변수 리스트
categorical_features = [
    'shelter_size_category',  # 소형/중형/대형/초대형
    'age_group',              # 자견/성견/노령견
    'sex_neutered',           # M_Y, M_N, F_Y, F_N, etc
    'province',               # 경기도/부산광역시
    'city',                   # 시/군/구
    'careNm'                  # 보호소 이름
]

# 방법 1: Label Encoding (트리 기반 모델용)
X_encoded = X.copy()
label_encoders = {}

for col in categorical_features:
    if col in X_encoded.columns:
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
        label_encoders[col] = le

# 방법 2: One-Hot Encoding (선형 모델용)
X_onehot = pd.get_dummies(
    X, 
    columns=categorical_features,
    drop_first=True  # 다중공선성 방지
)

print(f"\nLabel Encoding 후: {X_encoded.shape[1]}개 피처")
print(f"One-Hot Encoding 후: {X_onehot.shape[1]}개 피처")


Label Encoding 후: 20개 피처
One-Hot Encoding 후: 106개 피처


In [5]:
from sklearn.preprocessing import StandardScaler

# 스케일링이 필요한 수치형 피처
numeric_features = [
    'shelter_total_count',
    'shelter_recent_30d_load',
    'stay_duration_days',
    'health_sentiment_score',
    'notice_period_days',
    'weight_kg',
    'crowding_health_risk',
    'golden_attack_risk',
    'total_risk_score'
]

# 트리 기반 모델은 스케일링 불필요하지만, 
# 선형 모델/신경망에는 필수!

scaler = StandardScaler()
X_scaled = X_encoded.copy()
X_scaled[numeric_features] = scaler.fit_transform(X_encoded[numeric_features])

In [6]:
from sklearn.model_selection import train_test_split

# Train : Valid : Test = 70 : 15 : 15
X_train, X_temp, y_train, y_temp = train_test_split(
    X_encoded,  # 또는 X_onehot, X_scaled
    y, 
    test_size=0.3, 
    stratify=y,  # 클래스 비율 유지
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, 
    y_temp, 
    test_size=0.5, 
    stratify=y_temp,
    random_state=42
)

print(f"Train: {len(X_train)}건")
print(f"Valid: {len(X_val)}건")
print(f"Test:  {len(X_test)}건")

print(f"\nTrain 클래스 분포:")
print(y_train.value_counts(normalize=True).sort_index())

Train: 1855건
Valid: 397건
Test:  398건

Train 클래스 분포:
target
0    0.608086
1    0.125606
2    0.266307
Name: proportion, dtype: float64


In [7]:
# 필수 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# sklearn 모델
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor #트리계열 - 의사결정나무
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# sklearn 유틸리티
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Gradient Boosting 라이브러리
try:
    import xgboost as xgb
    print("XGBoost 버전:", xgb.__version__)
except ImportError:
    print("XGBoost 설치 필요: pip install xgboost")

try:
    import lightgbm as lgb
    print("LightGBM 버전:", lgb.__version__)
except ImportError:
    print("LightGBM 설치 필요: pip install lightgbm")

try:
    import catboost as cb
    print("CatBoost 버전:", cb.__version__)
except ImportError:
    print("CatBoost 설치 필요: pip install catboost")

# 경고 무시
import warnings
warnings.filterwarnings('ignore')

# 랜덤 시드 고정
np.random.seed(42)

print("환경 설정 완료!")

XGBoost 버전: 3.1.3
LightGBM 버전: 4.6.0
CatBoost 버전: 1.2.8
환경 설정 완료!


In [8]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

print("=" * 80)
print("LightGBM 모델 학습 시작")
print("=" * 80)

# 데이터 확인
print(f"\n데이터 크기:")
print(f"  Train: {X_train.shape}")
print(f"  Valid: {X_val.shape}")
print(f"  Test:  {X_test.shape}")

# 모델 초기화 (안정성 우선)
model_lgb = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1,  # ✅ 중요: 멀티스레드 문제 방지
    verbose=-1,
    force_col_wise=True  # ✅ 메모리 최적화
)

try:
    # 학습 (타임아웃 추가)
    print("\n학습 시작... (예상 시간: 1-2분)")
    
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=20, verbose=True),
            lgb.log_evaluation(period=10)
        ]
    )
    
    print(f"\n✓ 학습 완료!")
    print(f"  Best iteration: {model_lgb.best_iteration_}")
    print(f"  Best score: {model_lgb.best_score_['valid_0']['multi_logloss']:.4f}")
    
    # 평가
    y_pred_lgb = model_lgb.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred_lgb)
    
    print(f"\n{'=' * 80}")
    print(f"LightGBM Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"{'=' * 80}")
    
    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred_lgb,
        target_names=['생존', '자연사', '안락사'],
        digits=4
    ))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_lgb)
    cm_df = pd.DataFrame(
        cm,
        index=['실제: 생존', '실제: 자연사', '실제: 안락사'],
        columns=['예측: 생존', '예측: 자연사', '예측: 안락사']
    )
    print(cm_df)
    
    # XGBoost와 비교
    print("\n" + "=" * 80)
    print("모델 비교")
    print("=" * 80)
    print(f"XGBoost:  75.1%")
    print(f"LightGBM: {accuracy*100:.1f}%")
    
except KeyboardInterrupt:
    print("\n⚠️ 학습이 중단되었습니다.")
    print("다시 실행하거나 파라미터를 조정해주세요.")
    
except Exception as e:
    print(f"\n✗ 오류 발생: {e}")
    import traceback
    traceback.print_exc()

LightGBM 모델 학습 시작

데이터 크기:
  Train: (1855, 20)
  Valid: (397, 20)
  Test:  (398, 20)

학습 시작... (예상 시간: 1-2분)
Training until validation scores don't improve for 20 rounds
[10]	valid_0's multi_logloss: 0.640617
[20]	valid_0's multi_logloss: 0.586297
[30]	valid_0's multi_logloss: 0.569983
[40]	valid_0's multi_logloss: 0.563168
[50]	valid_0's multi_logloss: 0.563235
[60]	valid_0's multi_logloss: 0.566831
Early stopping, best iteration is:
[42]	valid_0's multi_logloss: 0.561652

✓ 학습 완료!
  Best iteration: 42
  Best score: 0.5617

LightGBM Accuracy: 0.7513 (75.13%)

Classification Report:
              precision    recall  f1-score   support

          생존     0.8348    0.7934    0.8136       242
         자연사     0.7586    0.4400    0.5570        50
         안락사     0.6115    0.8019    0.6939       106

    accuracy                         0.7513       398
   macro avg     0.7350    0.6784    0.6881       398
weighted avg     0.7658    0.7513    0.7494       398


Confusion Matrix:
         예

In [9]:
"""
complete_comparison.py - XGBoost vs LightGBM 완전 비교
"""

import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ============================================================================
# 1. 데이터 로드 및 전처리
# ============================================================================
print("=" * 80)
print("1. 데이터 로드 및 전처리")
print("=" * 80)

df = pd.read_csv('preprocessed_shelter_data.csv')
df_model = df[df['target'] >= 0].copy()

# 피처와 타겟 분리
exclude_cols = ['desertionNo', 'happenDt', 'processState', 'specialMark', 
                'noticeSdt', 'noticeEdt', 'updTm']
exclude_cols_existing = [col for col in exclude_cols if col in df_model.columns]

X = df_model.drop(['target'] + exclude_cols_existing, axis=1)
y = df_model['target']

# 카테고리 인코딩
categorical_features = ['shelter_size_category', 'age_group', 'sex_neutered', 
                       'province', 'city', 'careNm']
categorical_features_existing = [col for col in categorical_features if col in X.columns]

X_encoded = X.copy()
for col in categorical_features_existing:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))

# 데이터 분할
X_train, X_temp, y_train, y_temp = train_test_split(
    X_encoded, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train: {len(X_train)}건")
print(f"Valid: {len(X_val)}건")
print(f"Test:  {len(X_test)}건")

# ============================================================================
# 2. XGBoost 모델 학습
# ============================================================================
print("\n" + "=" * 80)
print("2. XGBoost 모델 학습")
print("=" * 80)

model_xgb = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    early_stopping_rounds=20
)

print("XGBoost 학습 시작...")
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=10
)

print(f"\n✓ XGBoost 학습 완료!")
print(f"  Best iteration: {model_xgb.best_iteration}")
print(f"  Best score: {model_xgb.best_score:.4f}")

# XGBoost 예측
y_pred_xgb = model_xgb.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

# ============================================================================
# 3. LightGBM 모델 학습
# ============================================================================
print("\n" + "=" * 80)
print("3. LightGBM 모델 학습")
print("=" * 80)

model_lgb = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1,
    verbose=-1,
    force_col_wise=True
)

print("LightGBM 학습 시작...")
model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20, verbose=True),
        lgb.log_evaluation(period=10)
    ]
)

print(f"\n✓ LightGBM 학습 완료!")
print(f"  Best iteration: {model_lgb.best_iteration_}")
print(f"  Best score: {model_lgb.best_score_['valid_0']['multi_logloss']:.4f}")

# LightGBM 예측
y_pred_lgb = model_lgb.predict(X_test)
lgb_accuracy = accuracy_score(y_test, y_pred_lgb)

# ============================================================================
# 4. 상세 비교
# ============================================================================
print("\n" + "=" * 80)
print("4. 모델 성능 비교")
print("=" * 80)

# 성능 비교 표
comparison_data = {
    '지표': ['Accuracy', 'Best Iteration', 'Best Score'],
    'XGBoost': [
        f'{xgb_accuracy:.4f} ({xgb_accuracy*100:.1f}%)',
        f'{model_xgb.best_iteration}',
        f'{model_xgb.best_score:.4f}'
    ],
    'LightGBM': [
        f'{lgb_accuracy:.4f} ({lgb_accuracy*100:.1f}%)',
        f'{model_lgb.best_iteration_}',
        f'{model_lgb.best_score_["valid_0"]["multi_logloss"]:.4f}'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n성능 비교:")
print(comparison_df.to_string(index=False))

# 예측 차이 분석
different_predictions = (y_pred_xgb != y_pred_lgb)
num_different = different_predictions.sum()

print(f"\n예측 일치도:")
print(f"  전체: {len(y_test)}개")
print(f"  일치: {len(y_test) - num_different}개 ({(1 - num_different/len(y_test))*100:.1f}%)")
print(f"  불일치: {num_different}개 ({num_different/len(y_test)*100:.1f}%)")

# 혼동 행렬 비교
print("\n[XGBoost 혼동 행렬]")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_xgb_df = pd.DataFrame(
    cm_xgb,
    index=['실제: 생존', '실제: 자연사', '실제: 안락사'],
    columns=['예측: 생존', '예측: 자연사', '예측: 안락사']
)
print(cm_xgb_df)

print("\n[LightGBM 혼동 행렬]")
cm_lgb = confusion_matrix(y_test, y_pred_lgb)
cm_lgb_df = pd.DataFrame(
    cm_lgb,
    index=['실제: 생존', '실제: 자연사', '실제: 안락사'],
    columns=['예측: 생존', '예측: 자연사', '예측: 안락사']
)
print(cm_lgb_df)

# Classification Report 비교
print("\n[XGBoost Classification Report]")
print(classification_report(
    y_test, y_pred_xgb,
    target_names=['생존', '자연사', '안락사'],
    digits=4
))

print("\n[LightGBM Classification Report]")
print(classification_report(
    y_test, y_pred_lgb,
    target_names=['생존', '자연사', '안락사'],
    digits=4
))

# ============================================================================
# 5. 최종 결론
# ============================================================================
print("\n" + "=" * 80)
print("5. 최종 결론")
print("=" * 80)

if abs(xgb_accuracy - lgb_accuracy) < 0.001:
    print(f"✓ 두 모델의 성능이 거의 동일합니다!")
    print(f"  XGBoost:  {xgb_accuracy:.4f}")
    print(f"  LightGBM: {lgb_accuracy:.4f}")
    print(f"\n✓ LightGBM이 더 빠르게 학습:")
    print(f"  XGBoost:  {model_xgb.best_iteration}회")
    print(f"  LightGBM: {model_lgb.best_iteration_}회")
    print(f"\n권장: 앙상블 (XGB + LGB) 또는 둘 중 편한 것 선택")
else:
    if xgb_accuracy > lgb_accuracy:
        print(f"✓ XGBoost가 더 우수합니다!")
        print(f"  XGBoost:  {xgb_accuracy:.4f}")
        print(f"  LightGBM: {lgb_accuracy:.4f}")
        print(f"  차이: {(xgb_accuracy - lgb_accuracy)*100:.2f}%p")
    else:
        print(f"✓ LightGBM이 더 우수합니다!")
        print(f"  XGBoost:  {xgb_accuracy:.4f}")
        print(f"  LightGBM: {lgb_accuracy:.4f}")
        print(f"  차이: {(lgb_accuracy - xgb_accuracy)*100:.2f}%p")

print("=" * 80)

1. 데이터 로드 및 전처리
Train: 1855건
Valid: 397건
Test:  398건

2. XGBoost 모델 학습
XGBoost 학습 시작...
[0]	validation_0-mlogloss:0.94555
[10]	validation_0-mlogloss:0.69679
[20]	validation_0-mlogloss:0.61234
[30]	validation_0-mlogloss:0.58690
[40]	validation_0-mlogloss:0.57540
[50]	validation_0-mlogloss:0.56799
[60]	validation_0-mlogloss:0.56898
[70]	validation_0-mlogloss:0.56954
[72]	validation_0-mlogloss:0.57098

✓ XGBoost 학습 완료!
  Best iteration: 52
  Best score: 0.5675

3. LightGBM 모델 학습
LightGBM 학습 시작...


Training until validation scores don't improve for 20 rounds
[10]	valid_0's multi_logloss: 0.640617
[20]	valid_0's multi_logloss: 0.586297
[30]	valid_0's multi_logloss: 0.569983
[40]	valid_0's multi_logloss: 0.563168
[50]	valid_0's multi_logloss: 0.563235
[60]	valid_0's multi_logloss: 0.566831
Early stopping, best iteration is:
[42]	valid_0's multi_logloss: 0.561652

✓ LightGBM 학습 완료!
  Best iteration: 42
  Best score: 0.5617

4. 모델 성능 비교

성능 비교:
            지표        XGBoost       LightGBM
      Accuracy 0.7513 (75.1%) 0.7513 (75.1%)
Best Iteration             52             42
    Best Score         0.5675         0.5617

예측 일치도:
  전체: 398개
  일치: 377개 (94.7%)
  불일치: 21개 (5.3%)

[XGBoost 혼동 행렬]
         예측: 생존  예측: 자연사  예측: 안락사
실제: 생존      189        8       45
실제: 자연사      20       24        6
실제: 안락사      17        3       86

[LightGBM 혼동 행렬]
         예측: 생존  예측: 자연사  예측: 안락사
실제: 생존      192        5       45
실제: 자연사      19       22        9
실제: 안락사      19        2       85

[XGB

In [10]:
"""
complete_model_comparison.py
XGBoost vs LightGBM vs Random Forest 완전 비교

작성자: 지성현
목적: 3개 주요 모델의 성능을 동일 조건에서 비교
"""

import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time

# ============================================================================
# 1. 데이터 로드 및 전처리
# ============================================================================
print("=" * 80)
print("Step 1: 데이터 로드 및 전처리")
print("=" * 80)

# 데이터 로드
df = pd.read_csv('preprocessed_shelter_data.csv')
df_model = df[df['target'] >= 0].copy()

print(f"전체 데이터: {len(df_model)}건")

# 피처와 타겟 분리
exclude_cols = ['desertionNo', 'happenDt', 'processState', 'specialMark', 
                'noticeSdt', 'noticeEdt', 'updTm']
exclude_cols_existing = [col for col in exclude_cols if col in df_model.columns]

X = df_model.drop(['target'] + exclude_cols_existing, axis=1)
y = df_model['target']

print(f"피처 수: {X.shape[1]}개")

# 카테고리 인코딩
categorical_features = ['shelter_size_category', 'age_group', 'sex_neutered', 
                       'province', 'city', 'careNm']
categorical_features_existing = [col for col in categorical_features if col in X.columns]

X_encoded = X.copy()
label_encoders = {}

print(f"\n카테고리 인코딩: {len(categorical_features_existing)}개")
for col in categorical_features_existing:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    label_encoders[col] = le
    print(f"  ✓ {col}")

# 데이터 분할
X_train, X_temp, y_train, y_temp = train_test_split(
    X_encoded, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"\n데이터 분할:")
print(f"  Train: {len(X_train):4d}건 ({len(X_train)/len(X_encoded)*100:.1f}%)")
print(f"  Valid: {len(X_val):4d}건 ({len(X_val)/len(X_encoded)*100:.1f}%)")
print(f"  Test:  {len(X_test):4d}건 ({len(X_test)/len(X_encoded)*100:.1f}%)")

# ============================================================================
# 2. Model 1: XGBoost
# ============================================================================
print("\n" + "=" * 80)
print("Step 2: XGBoost 모델 학습")
print("=" * 80)

start_time = time.time()

model_xgb = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    early_stopping_rounds=20
)

print("학습 중...")
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_time = time.time() - start_time

print(f"✓ 학습 완료! (소요 시간: {xgb_time:.2f}초)")
print(f"  Best iteration: {model_xgb.best_iteration}")
print(f"  Best score: {model_xgb.best_score:.4f}")

# 예측
y_pred_xgb = model_xgb.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
print(f"  Test Accuracy: {xgb_accuracy:.4f} ({xgb_accuracy*100:.1f}%)")

# ============================================================================
# 3. Model 2: LightGBM
# ============================================================================
print("\n" + "=" * 80)
print("Step 3: LightGBM 모델 학습")
print("=" * 80)

start_time = time.time()

model_lgb = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1,
    verbose=-1,
    force_col_wise=True
)

print("학습 중...")
model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20, verbose=False),
        lgb.log_evaluation(period=0)
    ]
)

lgb_time = time.time() - start_time

print(f"✓ 학습 완료! (소요 시간: {lgb_time:.2f}초)")
print(f"  Best iteration: {model_lgb.best_iteration_}")
print(f"  Best score: {model_lgb.best_score_['valid_0']['multi_logloss']:.4f}")

# 예측
y_pred_lgb = model_lgb.predict(X_test)
lgb_accuracy = accuracy_score(y_test, y_pred_lgb)
print(f"  Test Accuracy: {lgb_accuracy:.4f} ({lgb_accuracy*100:.1f}%)")

# ============================================================================
# 4. Model 3: Random Forest
# ============================================================================
print("\n" + "=" * 80)
print("Step 4: Random Forest 모델 학습")
print("=" * 80)

start_time = time.time()

model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

print("학습 중...")
model_rf.fit(X_train, y_train)

rf_time = time.time() - start_time

print(f"✓ 학습 완료! (소요 시간: {rf_time:.2f}초)")

# 예측
y_pred_rf = model_rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred_rf)
print(f"  Test Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.1f}%)")

# ============================================================================
# 5. 성능 비교
# ============================================================================
print("\n" + "=" * 80)
print("Step 5: 모델 성능 종합 비교")
print("=" * 80)

# 비교 테이블
comparison_data = {
    '모델': ['XGBoost', 'LightGBM', 'Random Forest'],
    'Accuracy': [
        f'{xgb_accuracy:.4f}',
        f'{lgb_accuracy:.4f}',
        f'{rf_accuracy:.4f}'
    ],
    'Accuracy (%)': [
        f'{xgb_accuracy*100:.1f}%',
        f'{lgb_accuracy*100:.1f}%',
        f'{rf_accuracy*100:.1f}%'
    ],
    '학습 시간 (초)': [
        f'{xgb_time:.2f}',
        f'{lgb_time:.2f}',
        f'{rf_time:.2f}'
    ],
    'Best Iteration': [
        model_xgb.best_iteration,
        model_lgb.best_iteration_,
        'N/A'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n[성능 비교표]")
print(comparison_df.to_string(index=False))

# 순위 매기기
accuracies = [xgb_accuracy, lgb_accuracy, rf_accuracy]
model_names = ['XGBoost', 'LightGBM', 'Random Forest']
sorted_indices = sorted(range(len(accuracies)), key=lambda i: accuracies[i], reverse=True)

print("\n[성능 순위]")
for rank, idx in enumerate(sorted_indices, 1):
    medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉"
    print(f"{medal} {rank}위: {model_names[idx]} - {accuracies[idx]*100:.1f}%")

# ============================================================================
# 6. 혼동 행렬 비교
# ============================================================================
print("\n" + "=" * 80)
print("Step 6: 혼동 행렬 비교")
print("=" * 80)

# XGBoost
print("\n[XGBoost 혼동 행렬]")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_xgb_df = pd.DataFrame(
    cm_xgb,
    index=['실제: 생존', '실제: 자연사', '실제: 안락사'],
    columns=['예측: 생존', '예측: 자연사', '예측: 안락사']
)
print(cm_xgb_df)

# LightGBM
print("\n[LightGBM 혼동 행렬]")
cm_lgb = confusion_matrix(y_test, y_pred_lgb)
cm_lgb_df = pd.DataFrame(
    cm_lgb,
    index=['실제: 생존', '실제: 자연사', '실제: 안락사'],
    columns=['예측: 생존', '예측: 자연사', '예측: 안락사']
)
print(cm_lgb_df)

# Random Forest
print("\n[Random Forest 혼동 행렬]")
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_rf_df = pd.DataFrame(
    cm_rf,
    index=['실제: 생존', '실제: 자연사', '실제: 안락사'],
    columns=['예측: 생존', '예측: 자연사', '예측: 안락사']
)
print(cm_rf_df)

# ============================================================================
# 7. Classification Report 비교
# ============================================================================
print("\n" + "=" * 80)
print("Step 7: 클래스별 성능 비교")
print("=" * 80)

print("\n[XGBoost Classification Report]")
print(classification_report(
    y_test, y_pred_xgb,
    target_names=['생존', '자연사', '안락사'],
    digits=4
))

print("\n[LightGBM Classification Report]")
print(classification_report(
    y_test, y_pred_lgb,
    target_names=['생존', '자연사', '안락사'],
    digits=4
))

print("\n[Random Forest Classification Report]")
print(classification_report(
    y_test, y_pred_rf,
    target_names=['생존', '자연사', '안락사'],
    digits=4
))

# ============================================================================
# 8. 피처 중요도 비교
# ============================================================================
print("\n" + "=" * 80)
print("Step 8: 피처 중요도 비교")
print("=" * 80)

# XGBoost 피처 중요도
feature_importance_xgb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_xgb.feature_importances_
}).sort_values('importance', ascending=False)

# LightGBM 피처 중요도
feature_importance_lgb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_lgb.feature_importances_
}).sort_values('importance', ascending=False)

# Random Forest 피처 중요도
feature_importance_rf = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n[XGBoost Top 10 피처]")
print(feature_importance_xgb.head(10).to_string(index=False))

print("\n[LightGBM Top 10 피처]")
print(feature_importance_lgb.head(10).to_string(index=False))

print("\n[Random Forest Top 10 피처]")
print(feature_importance_rf.head(10).to_string(index=False))

# 공통 상위 피처
top_5_xgb = set(feature_importance_xgb.head(5)['feature'].values)
top_5_lgb = set(feature_importance_lgb.head(5)['feature'].values)
top_5_rf = set(feature_importance_rf.head(5)['feature'].values)

common_features = top_5_xgb & top_5_lgb & top_5_rf

print(f"\n[3개 모델 공통 Top 5 피처]")
if common_features:
    for feature in common_features:
        print(f"  ✓ {feature}")
else:
    print("  (공통 피처 없음)")

# ============================================================================
# 9. 예측 일치도 분석
# ============================================================================
print("\n" + "=" * 80)
print("Step 9: 예측 일치도 분석")
print("=" * 80)

# XGBoost vs LightGBM
agreement_xgb_lgb = (y_pred_xgb == y_pred_lgb).sum()
print(f"\nXGBoost vs LightGBM:")
print(f"  일치: {agreement_xgb_lgb}/{len(y_test)} ({agreement_xgb_lgb/len(y_test)*100:.1f}%)")
print(f"  불일치: {len(y_test) - agreement_xgb_lgb}/{len(y_test)} ({(len(y_test) - agreement_xgb_lgb)/len(y_test)*100:.1f}%)")

# XGBoost vs Random Forest
agreement_xgb_rf = (y_pred_xgb == y_pred_rf).sum()
print(f"\nXGBoost vs Random Forest:")
print(f"  일치: {agreement_xgb_rf}/{len(y_test)} ({agreement_xgb_rf/len(y_test)*100:.1f}%)")
print(f"  불일치: {len(y_test) - agreement_xgb_rf}/{len(y_test)} ({(len(y_test) - agreement_xgb_rf)/len(y_test)*100:.1f}%)")

# LightGBM vs Random Forest
agreement_lgb_rf = (y_pred_lgb == y_pred_rf).sum()
print(f"\nLightGBM vs Random Forest:")
print(f"  일치: {agreement_lgb_rf}/{len(y_test)} ({agreement_lgb_rf/len(y_test)*100:.1f}%)")
print(f"  불일치: {len(y_test) - agreement_lgb_rf}/{len(y_test)} ({(len(y_test) - agreement_lgb_rf)/len(y_test)*100:.1f}%)")

# 3개 모델 모두 일치
all_agree = ((y_pred_xgb == y_pred_lgb) & (y_pred_lgb == y_pred_rf)).sum()
print(f"\n3개 모델 모두 일치:")
print(f"  {all_agree}/{len(y_test)} ({all_agree/len(y_test)*100:.1f}%)")

# ============================================================================
# 10. 최종 결론 및 권장사항
# ============================================================================
print("\n" + "=" * 80)
print("Step 10: 최종 결론 및 권장사항")
print("=" * 80)

# 최고 성능 모델
best_idx = sorted_indices[0]
best_model = model_names[best_idx]
best_accuracy = accuracies[best_idx]

print(f"\n🏆 최고 성능 모델: {best_model}")
print(f"   Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.1f}%)")

# 성능 차이
acc_diff_max = max(accuracies) - min(accuracies)
print(f"\n📊 모델 간 성능 차이: {acc_diff_max*100:.2f}%p")

if acc_diff_max < 0.01:
    print("   → 거의 동일한 성능! (차이 1% 미만)")
    print("   → 권장: 앙상블 또는 학습 속도가 빠른 LightGBM")
elif acc_diff_max < 0.03:
    print("   → 비슷한 성능 (차이 3% 미만)")
    print(f"   → 권장: {best_model} 사용")
else:
    print("   → 명확한 성능 차이")
    print(f"   → 권장: {best_model} 사용")

# 학습 속도 비교
fastest_idx = [xgb_time, lgb_time, rf_time].index(min([xgb_time, lgb_time, rf_time]))
print(f"\n⚡ 가장 빠른 모델: {model_names[fastest_idx]} ({[xgb_time, lgb_time, rf_time][fastest_idx]:.2f}초)")

# 앙상블 제안
print("\n💡 다음 단계 제안:")
print("   1. 앙상블 (Voting/Stacking):")
print(f"      XGBoost + LightGBM + Random Forest")
print(f"      예상 성능: {max(accuracies)*100:.1f}% → {(max(accuracies) + 0.02)*100:.1f}%")
print("   2. 하이퍼파라미터 튜닝 (Optuna):")
print(f"      예상 성능: {max(accuracies)*100:.1f}% → {(max(accuracies) + 0.03)*100:.1f}%")
print("   3. SMOTE (클래스 불균형 해결):")
print("      자연사 F1-Score 개선 기대")

print("\n" + "=" * 80)
print("분석 완료! 🎉")
print("=" * 80)

Step 1: 데이터 로드 및 전처리
전체 데이터: 2650건
피처 수: 20개

카테고리 인코딩: 6개
  ✓ shelter_size_category
  ✓ age_group
  ✓ sex_neutered
  ✓ province
  ✓ city
  ✓ careNm

데이터 분할:
  Train: 1855건 (70.0%)
  Valid:  397건 (15.0%)
  Test:   398건 (15.0%)

Step 2: XGBoost 모델 학습
학습 중...
✓ 학습 완료! (소요 시간: 0.11초)
  Best iteration: 52
  Best score: 0.5675
  Test Accuracy: 0.7513 (75.1%)

Step 3: LightGBM 모델 학습
학습 중...


✓ 학습 완료! (소요 시간: 0.07초)
  Best iteration: 42
  Best score: 0.5617
  Test Accuracy: 0.7513 (75.1%)

Step 4: Random Forest 모델 학습
학습 중...
✓ 학습 완료! (소요 시간: 0.36초)
  Test Accuracy: 0.7236 (72.4%)

Step 5: 모델 성능 종합 비교

[성능 비교표]
           모델 Accuracy Accuracy (%) 학습 시간 (초) Best Iteration
      XGBoost   0.7513        75.1%      0.11             52
     LightGBM   0.7513        75.1%      0.07             42
Random Forest   0.7236        72.4%      0.36            N/A

[성능 순위]
🥇 1위: XGBoost - 75.1%
🥈 2위: LightGBM - 75.1%
🥉 3위: Random Forest - 72.4%

Step 6: 혼동 행렬 비교

[XGBoost 혼동 행렬]
         예측: 생존  예측: 자연사  예측: 안락사
실제: 생존      189        8       45
실제: 자연사      20       24        6
실제: 안락사      17        3       86

[LightGBM 혼동 행렬]
         예측: 생존  예측: 자연사  예측: 안락사
실제: 생존      192        5       45
실제: 자연사      19       22        9
실제: 안락사      19        2       85

[Random Forest 혼동 행렬]
         예측: 생존  예측: 자연사  예측: 안락사
실제: 생존      185        5       52
실제: 자연사      21       17       12
실제

In [11]:
# 성능 비교표만 출력
print("\n" + "=" * 80)
print("성능 비교 요약")
print("=" * 80)
print(comparison_df)

# 순위
print("\n[성능 순위]")
for rank, idx in enumerate(sorted_indices, 1):
    medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉"
    print(f"{medal} {rank}위: {model_names[idx]} - {accuracies[idx]*100:.1f}%")

# 최종 결론
print("\n" + "=" * 80)
print("최종 결론")
print("=" * 80)
print(f"🏆 최고 성능 모델: {best_model}")
print(f"   Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.1f}%)")


성능 비교 요약
              모델 Accuracy Accuracy (%) 학습 시간 (초) Best Iteration
0        XGBoost   0.7513        75.1%      0.11             52
1       LightGBM   0.7513        75.1%      0.07             42
2  Random Forest   0.7236        72.4%      0.36            N/A

[성능 순위]
🥇 1위: XGBoost - 75.1%
🥈 2위: LightGBM - 75.1%
🥉 3위: Random Forest - 72.4%

최종 결론
🏆 최고 성능 모델: XGBoost
   Accuracy: 0.7513 (75.1%)


In [12]:
# # SMOTE 적용 코드 (바로 실행 가능)
# from imblearn.over_sampling import SMOTE

# print("SMOTE 적용 전:")
# print(f"  생존: {(y_train == 0).sum()}건")
# print(f"  자연사: {(y_train == 1).sum()}건")
# print(f"  안락사: {(y_train == 2).sum()}건")

# # SMOTE
# smote = SMOTE(random_state=42)
# X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# print("\nSMOTE 적용 후:")
# print(f"  생존: {(y_train_smote == 0).sum()}건")
# print(f"  자연사: {(y_train_smote == 1).sum()}건")
# print(f"  안락사: {(y_train_smote == 2).sum()}건")

# # XGBoost 재학습
# model_xgb_smote = xgb.XGBClassifier(
#     objective='multi:softmax',
#     num_class=3,
#     max_depth=6,
#     learning_rate=0.1,
#     n_estimators=200,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42,
#     eval_metric='mlogloss',
#     early_stopping_rounds=20
# )

# model_xgb_smote.fit(
#     X_train_smote, y_train_smote,
#     eval_set=[(X_val, y_val)],
#     verbose=10
# )

# # 평가
# y_pred_smote = model_xgb_smote.predict(X_test)
# print(f"\nSMOTE 적용 후 Accuracy: {accuracy_score(y_test, y_pred_smote):.4f}")
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred_smote, target_names=['생존', '자연사', '안락사']))

In [13]:
# from imblearn.over_sampling import SMOTE

# # 적절한 비율로 샘플링
# smote = SMOTE(
#     sampling_strategy={
#         0: 1128,  # 생존: 그대로
#         1: 600,   # 자연사: 233 → 600 (2.5배만 증가)
#         2: 800    # 안락사: 494 → 800 (1.6배 증가)
#     },
#     random_state=42
# )

# X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# print("조정된 SMOTE:")
# print(f"  생존: {(y_train_smote == 0).sum()}건")
# print(f"  자연사: {(y_train_smote == 1).sum()}건")
# print(f"  안락사: {(y_train_smote == 2).sum()}건")

# # 재학습
# model_xgb_smote = xgb.XGBClassifier(
#     objective='multi:softmax',
#     num_class=3,
#     max_depth=6,
#     learning_rate=0.1,
#     n_estimators=200,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42,
#     eval_metric='mlogloss',
#     early_stopping_rounds=20
# )

# model_xgb_smote.fit(
#     X_train_smote, y_train_smote,
#     eval_set=[(X_val, y_val)],
#     verbose=10
# )

# y_pred_smote = model_xgb_smote.predict(X_test)
# print(f"\n조정된 SMOTE Accuracy: {accuracy_score(y_test, y_pred_smote):.4f}")
# print(classification_report(y_test, y_pred_smote, target_names=['생존', '자연사', '안락사']))

In [14]:
# import optuna

# def objective(trial):
#     params = {
#         'max_depth': trial.suggest_int('max_depth', 4, 12),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#         'n_estimators': trial.suggest_int('n_estimators', 100, 500),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
#         'gamma': trial.suggest_float('gamma', 0, 0.5),
#     }
    
#     model = xgb.XGBClassifier(
#         objective='multi:softmax',
#         num_class=3,
#         random_state=42,
#         eval_metric='mlogloss',
#         early_stopping_rounds=20,
#         **params
#     )
    
#     model.fit(
#         X_train, y_train,
#         eval_set=[(X_val, y_val)],
#         verbose=False
#     )
    
#     y_pred = model.predict(X_val)
#     accuracy = accuracy_score(y_val, y_pred)
    
#     return accuracy

# # Optuna 최적화
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=50, show_progress_bar=True)

# print("최적 파라미터:")
# print(study.best_params)
# print(f"최고 Accuracy: {study.best_value:.4f}")

# # 최적 파라미터로 재학습
# best_model = xgb.XGBClassifier(
#     objective='multi:softmax',
#     num_class=3,
#     random_state=42,
#     **study.best_params
# )

# best_model.fit(X_train, y_train)
# y_pred_best = best_model.predict(X_test)

# print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
# print(classification_report(y_test, y_pred_best, target_names=['생존', '자연사', '안락사']))

In [15]:
# 이미 학습된 모델 사용 (early stopping 포함)
# 예측만 결합

# 각 모델의 확률 예측
y_pred_proba_xgb = model_xgb.predict_proba(X_test)
y_pred_proba_lgb = model_lgb.predict_proba(X_test)
y_pred_proba_rf = model_rf.predict_proba(X_test)

# 가중 평균 (XGB, LGB 더 높은 가중치)
y_pred_proba_ensemble = (
    2 * y_pred_proba_xgb + 
    2 * y_pred_proba_lgb + 
    1 * y_pred_proba_rf
) / 5

# 최종 예측
y_pred_ensemble = y_pred_proba_ensemble.argmax(axis=1)

# 평가
ensemble_accuracy = accuracy_score(y_test, y_pred_ensemble)

print(f"수동 앙상블 Test Accuracy: {ensemble_accuracy:.4f} ({ensemble_accuracy*100:.1f}%)")

print("\n성능 비교:")
print(f"XGBoost (원본):  75.1%")
print(f"LightGBM (원본): 75.1%")
print(f"Random Forest:   7X.X%")
print(f"수동 앙상블:      {ensemble_accuracy*100:.1f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ensemble, target_names=['생존', '자연사', '안락사']))

# 혼동 행렬
print("\n혼동 행렬:")
cm = confusion_matrix(y_test, y_pred_ensemble)
cm_df = pd.DataFrame(
    cm,
    index=['실제: 생존', '실제: 자연사', '실제: 안락사'],
    columns=['예측: 생존', '예측: 자연사', '예측: 안락사']
)
print(cm_df)

수동 앙상블 Test Accuracy: 0.7462 (74.6%)

성능 비교:
XGBoost (원본):  75.1%
LightGBM (원본): 75.1%
Random Forest:   7X.X%
수동 앙상블:      74.6%

Classification Report:
              precision    recall  f1-score   support

          생존       0.83      0.79      0.81       242
         자연사       0.70      0.42      0.53        50
         안락사       0.61      0.81      0.70       106

    accuracy                           0.75       398
   macro avg       0.72      0.67      0.68       398
weighted avg       0.76      0.75      0.74       398


혼동 행렬:
         예측: 생존  예측: 자연사  예측: 안락사
실제: 생존      190        7       45
실제: 자연사      20       21        9
실제: 안락사      18        2       86


In [24]:
print("SHAP Values 구조 확인:")
print(f"type(shap_values): {type(shap_values)}")
print(f"len(shap_values): {len(shap_values)}")

for i in range(len(shap_values)):
    print(f"shap_values[{i}] shape: {shap_values[i].shape}")
    print(f"shap_values[{i}] type: {type(shap_values[i])}")

print(f"\nX_test shape: {X_test_clean.shape}")
print(f"y_test shape: {y_test_clean.shape}")

SHAP Values 구조 확인:
type(shap_values): <class 'numpy.ndarray'>
len(shap_values): 398
shap_values[0] shape: (20, 3)
shap_values[0] type: <class 'numpy.ndarray'>
shap_values[1] shape: (20, 3)
shap_values[1] type: <class 'numpy.ndarray'>
shap_values[2] shape: (20, 3)
shap_values[2] type: <class 'numpy.ndarray'>
shap_values[3] shape: (20, 3)
shap_values[3] type: <class 'numpy.ndarray'>
shap_values[4] shape: (20, 3)
shap_values[4] type: <class 'numpy.ndarray'>
shap_values[5] shape: (20, 3)
shap_values[5] type: <class 'numpy.ndarray'>
shap_values[6] shape: (20, 3)
shap_values[6] type: <class 'numpy.ndarray'>
shap_values[7] shape: (20, 3)
shap_values[7] type: <class 'numpy.ndarray'>
shap_values[8] shape: (20, 3)
shap_values[8] type: <class 'numpy.ndarray'>
shap_values[9] shape: (20, 3)
shap_values[9] type: <class 'numpy.ndarray'>
shap_values[10] shape: (20, 3)
shap_values[10] type: <class 'numpy.ndarray'>
shap_values[11] shape: (20, 3)
shap_values[11] type: <class 'numpy.ndarray'>
shap_values[

In [25]:
import shap
import pandas as pd
import numpy as np

print("=" * 80)
print("SHAP 분석 (완벽 이해)")
print("=" * 80)

# 데이터 준비
X_test_clean = X_test.copy().reset_index(drop=True)
y_test_clean = y_test.copy().reset_index(drop=True) if hasattr(y_test, 'copy') else pd.Series(y_test).reset_index(drop=True)
y_pred_xgb_clean = model_xgb.predict(X_test_clean)

print(f"샘플: {len(X_test_clean)}")
print(f"피처: {X_test_clean.shape[1]}")

# SHAP 계산
print("\nSHAP 계산 중...")
explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X_test_clean.values)

print(f"✓ 완료")
print(f"  구조: {len(shap_values)}개 샘플")
print(f"  각 샘플: {shap_values[0].shape} (피처, 클래스)")

# ============================================================================
# 피처 중요도
# ============================================================================
print("\n피처 중요도:")

# 모든 샘플의 안락사 클래스(2) SHAP 값 수집
all_shap_class2 = []
for sample_idx in range(len(shap_values)):
    # shap_values[sample_idx]는 (20, 3)
    # [:, 2]로 안락사 클래스만 추출 → (20,)
    shap_for_class2 = shap_values[sample_idx][:, 2]
    all_shap_class2.append(shap_for_class2)

# (398, 20) 배열로 변환
shap_class2_array = np.array(all_shap_class2)

# 평균 절대값
mean_abs_shap = np.abs(shap_class2_array).mean(axis=0)
indices_sorted = np.argsort(mean_abs_shap)[::-1]

print("\n[안락사 예측 Top 10 피처]")
for rank, idx in enumerate(indices_sorted[:10], 1):
    print(f"  {rank:2d}. {X_test_clean.columns[idx]:30s} : {mean_abs_shap[idx]:.4f}")

# CSV 저장
with open('shap_importance.csv', 'w') as f:
    f.write("rank,feature,mean_abs_shap\n")
    for rank, idx in enumerate(indices_sorted, 1):
        f.write(f"{rank},{X_test_clean.columns[idx]},{mean_abs_shap[idx]:.6f}\n")

print("\n✓ 저장: shap_importance.csv")

# ============================================================================
# 클래스별 Top 5
# ============================================================================
print("\n클래스별 피처 중요도:")

for class_idx, class_name in enumerate(['생존', '자연사', '안락사']):
    all_shap_class = []
    for sample_idx in range(len(shap_values)):
        shap_for_class = shap_values[sample_idx][:, class_idx]
        all_shap_class.append(shap_for_class)
    
    shap_class_array = np.array(all_shap_class)
    mean_abs = np.abs(shap_class_array).mean(axis=0)
    indices = np.argsort(mean_abs)[::-1]
    
    print(f"\n[{class_name}]")
    for rank, idx in enumerate(indices[:5], 1):
        print(f"  {rank}. {X_test_clean.columns[idx]:30s} : {mean_abs[idx]:.4f}")

# ============================================================================
# 오분류 분석
# ============================================================================
print("\n" + "=" * 80)
print("오분류 케이스 분석")
print("=" * 80)

y_test_array = np.array(y_test_clean)

# 생존 -> 안락사
fp_mask = (y_test_array == 0) & (y_pred_xgb_clean == 2)
fp_indices = np.where(fp_mask)[0]

print(f"\n생존 -> 안락사: {len(fp_indices)}건")

if len(fp_indices) > 0:
    print("\n상위 3개:")
    
    for i, sample_idx in enumerate(fp_indices[:3]):
        print(f"\n  케이스 {i+1} (Index {sample_idx}):")
        print(f"    실제: 생존, 예측: 안락사")
        
        proba = model_xgb.predict_proba(X_test_clean.iloc[[sample_idx]])[0]
        print(f"    확률: 생존 {proba[0]:.3f}, 자연사 {proba[1]:.3f}, 안락사 {proba[2]:.3f}")
        
        # 안락사 클래스 SHAP 값 (20,)
        shap_for_sample = shap_values[sample_idx][:, 2]
        
        top_5_features = np.argsort(np.abs(shap_for_sample))[::-1][:5]
        
        print(f"    Top 5 영향 피처:")
        for j, feat_idx in enumerate(top_5_features, 1):
            feat_name = X_test_clean.columns[feat_idx]
            feat_value = X_test_clean.iloc[sample_idx, feat_idx]
            shap_val = shap_for_sample[feat_idx]
            print(f"      {j}. {feat_name:25s}: {feat_value:7.2f} (SHAP: {shap_val:+.4f})")
    
    # 공통 패턴
    print(f"\n  공통 패턴 ({len(fp_indices)}건):")
    
    fp_shap_list = []
    for sample_idx in fp_indices:
        fp_shap_list.append(shap_values[sample_idx][:, 2])
    
    fp_shap_array = np.array(fp_shap_list)  # (n, 20)
    avg_shap_fp = np.abs(fp_shap_array).mean(axis=0)  # (20,)
    top_5_common = np.argsort(avg_shap_fp)[::-1][:5]
    
    print("    주요 원인 피처:")
    for rank, feat_idx in enumerate(top_5_common, 1):
        print(f"      {rank}. {X_test_clean.columns[feat_idx]:30s}: {avg_shap_fp[feat_idx]:.4f}")

# 안락사 -> 생존
fn_mask = (y_test_array == 2) & (y_pred_xgb_clean == 0)
fn_indices = np.where(fn_mask)[0]

print(f"\n안락사 -> 생존: {len(fn_indices)}건")

if len(fn_indices) > 0:
    print("\n상위 3개:")
    
    for i, sample_idx in enumerate(fn_indices[:3]):
        print(f"\n  케이스 {i+1} (Index {sample_idx}):")
        print(f"    실제: 안락사, 예측: 생존")
        
        proba = model_xgb.predict_proba(X_test_clean.iloc[[sample_idx]])[0]
        print(f"    확률: 생존 {proba[0]:.3f}, 자연사 {proba[1]:.3f}, 안락사 {proba[2]:.3f}")
        
        # 생존 클래스 SHAP 값
        shap_for_sample = shap_values[sample_idx][:, 0]
        
        top_5_features = np.argsort(np.abs(shap_for_sample))[::-1][:5]
        
        print(f"    Top 5 영향 피처:")
        for j, feat_idx in enumerate(top_5_features, 1):
            feat_name = X_test_clean.columns[feat_idx]
            feat_value = X_test_clean.iloc[sample_idx, feat_idx]
            shap_val = shap_for_sample[feat_idx]
            print(f"      {j}. {feat_name:25s}: {feat_value:7.2f} (SHAP: {shap_val:+.4f})")
    
    # 공통 패턴
    print(f"\n  공통 패턴 ({len(fn_indices)}건):")
    
    fn_shap_list = []
    for sample_idx in fn_indices:
        fn_shap_list.append(shap_values[sample_idx][:, 0])
    
    fn_shap_array = np.array(fn_shap_list)
    avg_shap_fn = np.abs(fn_shap_array).mean(axis=0)
    top_5_common = np.argsort(avg_shap_fn)[::-1][:5]
    
    print("    주요 원인 피처:")
    for rank, feat_idx in enumerate(top_5_common, 1):
        print(f"      {rank}. {X_test_clean.columns[feat_idx]:30s}: {avg_shap_fn[feat_idx]:.4f}")

# ============================================================================
# 완료
# ============================================================================
print("\n" + "=" * 80)
print("SHAP 분석 완료! 🎉")
print("=" * 80)
print(f"\n주요 발견:")
print(f"  1. 가장 중요한 피처: {X_test_clean.columns[indices_sorted[0]]}")
print(f"  2. 생존->안락사: {len(fp_indices)}건")
print(f"  3. 안락사->생존: {len(fn_indices)}건")
print(f"\n파일: shap_importance.csv")
print("=" * 80)

SHAP 분석 (완벽 이해)
샘플: 398
피처: 20

SHAP 계산 중...
✓ 완료
  구조: 398개 샘플
  각 샘플: (20, 3) (피처, 클래스)

피처 중요도:

[안락사 예측 Top 10 피처]
   1. shelter_total_count            : 0.7371
   2. weight_kg                      : 0.3392
   3. is_mixed                       : 0.2000
   4. shelter_recent_30d_load        : 0.1232
   5. city                           : 0.0939
   6. careNm                         : 0.0925
   7. total_risk_score               : 0.0901
   8. crowding_health_risk           : 0.0801
   9. health_sentiment_score         : 0.0488
  10. notice_period_days             : 0.0358

✓ 저장: shap_importance.csv

클래스별 피처 중요도:

[생존]
  1. is_mixed                       : 0.5058
  2. shelter_total_count            : 0.2990
  3. weight_kg                      : 0.1649
  4. city                           : 0.1544
  5. health_sentiment_score         : 0.1256

[자연사]
  1. weight_kg                      : 0.2908
  2. city                           : 0.2086
  3. health_sentiment_score         : 0.1526
  4. to